In [3]:
import pandas as pd
from sklearn.linear_model import LinearRegression

In [4]:
# load data
df = pd.read_csv("cleaned_expenses.csv")


In [5]:

# convert date
df['date'] = pd.to_datetime(df['date'])

In [6]:
# create month column
df['month'] = df['date'].dt.to_period('M')

In [7]:
# group by category and month
grouped = df.groupby(['category', 'month'])['amount'].sum().reset_index()


In [8]:
# store predictions
results = {}


In [9]:
# loop through categories
for cat in grouped['category'].unique():
    
    # filter category data
    temp = grouped[grouped['category'] == cat].copy()
    
    # sort by month
    temp = temp.sort_values('month')
    
    # create index
    temp['month_index'] = range(len(temp))
    
    # features and target
    X = temp[['month_index']]
    y = temp['amount']
    
    # skip small data
    if len(temp) < 2:
        continue
    
    # train model
    model = LinearRegression()
    model.fit(X, y)
    
    # predict next month
    next_month = pd.DataFrame({'month_index': [len(temp)]})
    pred = max(0, model.predict(next_month)[0])
    
    # store result
    results[cat] = pred

In [10]:

# print results
print("Next month category-wise prediction:\n")
for k, v in results.items():
    print(k, "->", round(v, 2))

Next month category-wise prediction:

bills & fees -> 13044.96
food & drinks -> 2067.75
not defined -> 123.92
transport -> 1508.27


In [11]:
# save results
pred_df = pd.DataFrame(list(results.items()), columns=['category', 'predicted_amount'])
pred_df.to_csv("category_predictions.csv", index=False)
